# Modules tour: Predict through MultiChainComparison

This notebook walks through the core DSPy modules covered in Chapter 7: `Predict`, `ChainOfThought`, `BestOfN`, `Refine`, and `MultiChainComparison`. Each module wraps a Signature in a different inference strategy.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## 7.1.1 Predict

Predict is the foundational module. Every other module in this section delegates to it under the hood; earlier chapters cover its basic usage.


## 7.1.2 ChainOfThought with a custom rationale field

`ChainOfThought` prepends a reasoning field to your Signature. You can customize the reasoning field if the default prefix doesn't fit your task.


In [ ]:
custom_reasoning = dspy.OutputField(
    desc=(
        "Explain the clinical reasoning based on the patient's "
        "symptoms before assigning an urgency level."
    )
)

triage = dspy.ChainOfThought(
    "symptoms: str -> urgency: str",
    rationale_field=custom_reasoning
)

result = triage(
    symptoms="Patient reports sudden severe headache and stiff neck"
)

print(result.reasoning)
print(result.urgency)


## 7.1.3 BestOfN

`BestOfN` retries with a reward function and supports early stopping via the `threshold` parameter. This example gives a tagline writer up to five attempts to produce exactly six words.


In [ ]:
tagline_writer = dspy.ChainOfThought(
    "product: str -> tagline: str"
)

def word_count_check(args, pred):
    """Check whether the tagline contains exactly six words."""
    words = pred.tagline.strip().split()
    return 1.0 if len(words) == 6 else 0.0

best_tagline = dspy.BestOfN(
    module=tagline_writer,
    N=5,
    reward_fn=word_count_check,
    threshold=1.0
)

result = best_tagline(
    product="Noise-cancelling headphones for frequent travelers"
)


## 7.1.4 Refine

`Refine` has the same API as `BestOfN`, but generates targeted feedback for the next retry after an attempt misses the threshold.


In [ ]:
sql_writer = dspy.ChainOfThought(
    "question: str, db_schema: str -> sql_query: str"
)

def validate_sql(args, pred):
    """Validate the SQL query is syntactically correct and answers the question."""
    query = pred.sql_query.strip()
    if not query.upper().startswith("SELECT"):
        return 0.0
    if args["db_schema"].split(",")[0].strip() not in query:
        return 0.3
    return 1.0

refined_sql = dspy.Refine(
    module=sql_writer,
    N=4,
    reward_fn=validate_sql,
    threshold=1.0
)

result = refined_sql(
    question="How many orders were placed last month?",
    db_schema="orders (id, customer_id, placed_at, total), customers (id, name)"
)


## 7.1.5 MultiChainComparison

Generate several independent reasoning paths, then compare them to produce one final prediction without writing a reward function.


In [ ]:
signature = "incident: str -> root_cause: str"
investigate = dspy.ChainOfThought(signature)

incident = (
    "Checkout latency spiked after database CPU reached 95%. "
    "Application servers then began retrying failed requests."
)

# Generate three independent reasoning paths
attempts = investigate(
    incident=incident,
    config={"n": 3, "temperature": 1.0}
)

# Compare the three attempts and produce one final answer
compare = dspy.MultiChainComparison(signature, M=3, temperature=1.0)
result = compare(attempts.completions, incident=incident)

print(result.root_cause)
